# Claude Beer Game: student personalities and extra deliberation

Protocol **student-high-claude-revision-v4** uses **Claude Haiku 5.5, adaptive thinking, fixed high effort**, through the direct Anthropic API. The four fictional UB sophomore profiles, instructions, 24-week games, 12-week automatic scoring, token ceilings and H1/H2/H3 definitions match the GPT revision study. Claude receives its own fresh calibration. Results stay separate across models.

The prospective target is **8 paired blocks (16 chains)** if calibration passes and the measured cost forecast fits the cap. The new-run cap is **$2.15**; the shared redesign guard is **$2.95**, including the **$0.7877313225** already spent on Mini/Luna. Opening this notebook makes no paid calls. [Protocol](../docs/CLAUDE_PROTOCOL_20261011.md). Eight is an inference floor, not a guarantee of power or significance.


## Completed Claude replication ? 11 October 2026

[Claude API prompts](../reports/claude_api_fd5533fef5828d96/CLAUDE_API_PROMPTS.md): exact prompt wording in readable boxes, with short explanations of where each is used. The [GPT prompts](../reports/economy_api_a80275c58f2c07e0/GPT_API_PROMPTS.md) are in a separate file. Exported from saved evidence on 17 October 2026; no API calls.

**All 8 paired blocks completed: 16 chains, 1,280 model orders, zero fallbacks.** All 2,060 API responses completed. Calibration and actual-computation checks passed; median total generated tokens were **506 short versus 1,121 adequate**. This Claude run cost **$1.065919405**; cumulative redesign spending including earlier Mini/Luna calls is **$1.8536507275**, below $3.

Read the [findings](../reports/claude_api_fd5533fef5828d96/FINDINGS.md), [findings Word document](../reports/claude_api_fd5533fef5828d96/Pilot_Findings.docx), [full report](../reports/claude_api_fd5533fef5828d96/REPORT.md), [full Word report](../reports/claude_api_fd5533fef5828d96/Claude_Study_Report.docx), [workbook](../reports/claude_api_fd5533fef5828d96/DETAILS.xlsx), and [paired-difference figure](../reports/claude_api_fd5533fef5828d96/paired_contrasts.pdf).

- **Revised H1:** lower pipeline responsiveness under short deliberation; difference -0.102865, Holm-adjusted p=0.015987. This supports the matched-state probe hypothesis within the configured Claude setup.
- **H2 cost and primary H3 bullwhip:** inconclusive, both Holm-adjusted p=0.327260. Their point estimates are opposite to the predictions. The mean bullwhip ratios are 0.361597 short versus 0.435084 adequate; all chain averages are below 1.
- The cost contrast reverses between model weeks (+12.0625 per pair) and automatic continuation (-89.3125). The original structural-beta analysis has only one fully usable pair and remains unresolved. No human students were observed.

With all paid flags off and `WATCH_RUNNING=True`, the watcher now shows complete, 16/16 chains. The review cell reads saved evidence without a key. For a separately planned new run, choose a new replicate ID and set `CALIBRATION_RUN = ROOT / 'data/runs/claude_api_fd5533fef5828d96'` to reuse compatible calibration. Keep the campaign ID and all spending guards. The default seed repeats the indexed demand paths; additional draws do not automatically add new demand scenarios. Do not extend a sample based on whether its results are significant.


In [ ]:
from pathlib import Path
import sys, json, os, getpass, asyncio
from dataclasses import replace
import pandas as pd
from IPython.display import display, Markdown, Image
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src/beergame_study').exists():
    raise RuntimeError('Open this notebook from the beergame project.')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from beergame_study.claude_protocol import ClaudeConfig
from beergame_study.claude_study import run_study
from beergame_study.claude_report import analyze, build_report, planned_pairs
from beergame_study.student_profiles import profiles, assignment
from beergame_study.notebook_progress import NotebookProgress
from beergame_study.operational_io import install_windows_file_retry
install_windows_file_retry()
print('Project:', ROOT)

## Controls

Set a run flag to `True` only for the stage you intend to execute. Keep `MODE='offline'` for free software checks. Set `MODE='api'` explicitly for paid calibration/games. Calibration is 16 scenarios (32 calls at most), not full games. A failed live screen prevents purchasing games.

The 24-week setting is a prespecified inexpensive feasibility game: four practice weeks, 20 model weeks, then 12 automatic scoring weeks. Use 40 or more for a separately planned larger study. Choose the horizon and independent pair count before outcomes. Increase pairs in multiples of four to balance personality assignments across roles. Do not treat four roles or weekly calls as independent students.

In [ ]:
MODE = 'offline'
RUN_OFFLINE_CHECK = False
RUN_CALIBRATION = False
RUN_GAMES = False
WATCH_RUNNING = True
WEEKS = 24
PAIRS = 8
RUN_CAP_USD = 2.15
CAMPAIGN_ID = '20261011'  # Existing authorization includes all earlier spending.
REPLICATE_ID = 'claude-replication-01'  # New independent games require a new ID.
CALIBRATION_RUN = None  # Optional compatible completed data/runs/... directory.
REPORT_RUN = None       # Optional saved run for analysis; no key needed.
config = ClaudeConfig(mode=MODE, weeks=WEEKS, groups_per_condition=PAIRS,
    budget_usd=RUN_CAP_USD, campaign_id=CAMPAIGN_ID, replicate_id=REPLICATE_ID)
config.validate()
print('Model:', config.model, '| Effort: high | Temperature: omitted, held constant')
print('Short: initial <= 16384 tokens; adequate: same initial + revision <= 8192')
print('Actual combined tokens are checked; these are ceilings, not guaranteed usage.')
display(pd.DataFrame(profiles().values()))
display(pd.DataFrame([dict(pair=b, **dict(zip(('retailer','wholesaler','distributor','factory'), assignment(config.seed,b)))) for b in range(PAIRS)]))
print('Anthropic standard only; calibration cap $0.18; run cap $2.15; prior redesign spending $0.7877313225.')
print('The four-state quick check precedes full held-out calibration; failed gates buy no games.')

In [ ]:
async def new_client():
    if MODE != 'api':
        return None
    from anthropic import AsyncAnthropic
    key = os.environ.get('ANTHROPIC_API_KEY') or getpass.getpass('Anthropic API key (hidden; memory only): ')
    if not key or not key.strip():
        raise RuntimeError('No key entered; no paid calls made.')
    client = AsyncAnthropic(base_url='https://api.anthropic.com', api_key=key.strip(), max_retries=0, timeout=config.timeout_s)
    key = None
    return client

if RUN_OFFLINE_CHECK:
    offline_config = replace(config, mode='offline', replicate_id=REPLICATE_ID+'-software')
    OFFLINE_RUN = await run_study(offline_config, root=ROOT, on_progress=NotebookProgress())
    print('SYNTHETIC software report:', ROOT/'reports'/OFFLINE_RUN.name/'REPORT.md')
else:
    print('Offline campaign not started. No calls made.')

## Calibration

This measures initial-choice completion, completed revisions, and actual total computation on early/middle/late histories and all four profiles. It never selects settings using order sizes or hypothesis direction. The progress bar below displays stage, active requests and actual charges/reservations.

In [ ]:
if RUN_CALIBRATION:
    client = await new_client()
    try:
        CALIBRATION_RUN = await run_study(config, client, root=ROOT, calibration_only=True,
                                         on_progress=NotebookProgress())
    finally:
        if client is not None:
            await client.close()
    display(Markdown((ROOT/'reports'/CALIBRATION_RUN.name/'REPORT.md').read_text(encoding='utf-8')))
else:
    print('Calibration not started. Set RUN_CALIBRATION=True to run the selected MODE.')

## Paired games

A compatible saved calibration is reused without new calibration purchases. The exact full initial prompt is shared only within a paired week/role when both histories match. Once histories differ, each condition buys its own initial choice. No reuse across independent pairs. A forecast and per-request dollar guard limit spending; budget stops still produce partial reports.

In [ ]:
if RUN_GAMES:
    client = await new_client()
    try:
        GAME_RUN = await run_study(config, client, root=ROOT,
            calibration_run=CALIBRATION_RUN, on_progress=NotebookProgress())
        REPORT_RUN = GAME_RUN
    finally:
        if client is not None:
            await client.close()
    print('Report folder:', ROOT/'reports'/GAME_RUN.name)
else:
    print('Games not started. Set RUN_GAMES=True to run the selected MODE.')

## Watch a run started from PowerShell

Use this instead of the run cells when a background process is already active. It only reads progress. Interrupt the cell to stop watching; this does not cancel the experiment.

In [ ]:
if WATCH_RUNNING:
    render = NotebookProgress()
    while True:
        pointer = ROOT/'reports/ACTIVE_CLAUDE_RUN.json'
        if not pointer.exists():
            print('No revised run is active yet.')
            break
        active = json.loads(pointer.read_text(encoding='utf-8'))
        status = json.loads(Path(active['progress']).read_text(encoding='utf-8'))
        render(status)
        if status['phase'] in ('complete', 'stopped'):
            REPORT_RUN = Path(active['run'])
            break
        await asyncio.sleep(2)
else:
    print('Progress watcher disabled.')

## Review results and hypothesis checks (no API calls)

This displays the primary bullwhip result, cost result and matched-state supply-line sensitivity. The original structural beta fits remain exploratory. Unusable coefficients stay flagged. One pair cannot support an inferential conclusion. Synthetic results are explicitly labeled.

In [ ]:
if REPORT_RUN is None:
    pointer = ROOT/'reports/LATEST_CLAUDE_REPORT.json'
    if pointer.exists():
        REPORT_RUN = Path(json.loads(pointer.read_text(encoding='utf-8'))['run'])
if REPORT_RUN is not None and (Path(REPORT_RUN)/'progress.json').exists():
    status = json.loads((Path(REPORT_RUN)/'progress.json').read_text(encoding='utf-8'))
    if status['phase'] not in ('complete', 'stopped'):
        print('Run is active; use the watcher before exporting its report.')
    else:
        report_folder = build_report(REPORT_RUN, root=ROOT)
        result = analyze(REPORT_RUN)
        print('Report:', report_folder/'Claude_Study_Report.docx')
        print('Workbook:', report_folder/'DETAILS.xlsx')
        display(result['hypotheses'])
        display(result['groups'])
        display(result['roles'])
        if (report_folder/'orders.png').exists():
            display(Image(filename=str(report_folder/'orders.png')))
else:
    print('No revised run exists yet. The literature/design review is under reports/revised_design_20261011/.')

## Plan replication and costs

The following is an **illustration under statistical assumptions**, not a promise of significance or a recommended effect size. The effect is measured in standard deviations of independent paired-chain differences. Meaningful effects and variance estimates must justify the final sample plan. Eight pairs is only the software inference floor. A new campaign cap requires your explicit authorization.

In [ ]:
display(pd.DataFrame([planned_pairs(d) for d in (0.5, 0.8, 1.0)]))
print('Cost controls: within-pair initial sharing, prompt-prefix caching, exact resume, saved calibration reuse, free analysis.')
print('Batch API can be cheaper but is not implemented in this interactive runner; sequential games can be slow in batch.')